# 84-Doc Legal / News / Regulatory **Application Anchor** — Demo

This notebook is a runnable demo of the *application-anchor* dataset built for a
**text → first-order-logic → Prolog** neuro-symbolic atomic-fact-extraction &
hallucination-control experiment.

**What the dataset is.** 84 short, professionally-written documents
(legal 30 / news 28 / regulatory 26) standardized to a shared
`(head, relation, tail)` triple schema with **char-span provenance**, grouped by
source corpus:

| corpus | genre | gold quality | license |
|---|---|---|---|
| **CUAD** (30) | legal | **crisp** (human clause spans) | CC BY 4.0 |
| **Wikinews** (28) | news | silver (deterministic spaCy SVO+5W) | CC BY 2.5 |
| **GDPR** (15) | regulatory-EU | silver (structural regex) | EUR-Lex free reuse |
| **eCFR** (11) | regulatory-US | silver (structural regex) | US public domain |

Each example: `input` is a JSON **string**
`{doc_id, document_text, genre, source, char_length, entities:[{name,type,char_span}]}`
and `output` is a JSON **string** `{gold_atomic_facts:[{head,relation,tail,provenance_char_span}]}`.
**No LLM is used anywhere in gold construction** (non-circularity).

**What this demo does.** It loads a curated subset of the data from GitHub, re-groups
it exactly as `data.py` does, recomputes the dataset statistics with the original
`build_meta()` function (copied verbatim), re-verifies char-span provenance the way
`build/verify_dataset.py` does, and visualizes the genre / gold-quality / decidability
structure. The original builder (`data.py` + `build_{legal,news,regulatory}.py`)
regenerates the gold from a 40 MB raw snapshot using spaCy/nltk; that step is **not**
re-run here — the demo loads the already-built data and reproduces the analysis side.

In [ ]:
# --- Install dependencies (works on Colab and locally) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru -- NOT pre-installed on Colab, always install (data.py logs with loguru)
_pip('loguru==0.7.2')

# matplotlib (+ numpy) -- pre-installed on Colab; install locally at Colab's versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
# --- original data.py import block ---
# (the builder-only modules build_legal / build_news / build_regulatory are omitted:
#  this demo LOADS the pre-built dataset instead of regenerating it from the 40 MB
#  raw/ snapshot + spaCy/nltk models)
from __future__ import annotations
import os, sys, json, glob, statistics, hashlib
from pathlib import Path
from collections import Counter, defaultdict
from loguru import logger

# additional import for the demo's visualization cell
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# --- Data loading: GitHub URL with local fallback (Colab-compatible) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6db730-decoy-gated-neuro-symbolic-extraction-a/main/round-4/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
logger.info(f"loaded application-anchor demo: {len(data['datasets'])} source-corpus groups "
            f"({[(g['dataset'], len(g['examples'])) for g in data['datasets']]})")

## Configuration

All tunable demo parameters live here. They start **minimal** (smallest values that
still produce meaningful output) and can be scaled up.

- `MAX_DOCS_PER_CORPUS` — documents loaded per source corpus (the mini demo file ships
  5 per corpus; the **full** dataset has up to 30/corpus).
- `N_WORKED_EXAMPLES` — how many documents to inspect in full detail (text + entities + gold facts).

The block below also re-declares the builder constants (`SEED`, `CORPUS_ORDER`,
`TOOL_VERSIONS`) copied verbatim from `data.py`.

In [ ]:
# === Tunable demo parameters (START MINIMAL; scale up after it runs) ===
MAX_DOCS_PER_CORPUS = 5      # mini demo file ships <=5 per corpus; full dataset has up to 30
N_WORKED_EXAMPLES   = 2      # documents shown in full detail

# === Builder constants copied verbatim from data.py ===
SEED = 42
# stable source-corpus -> dataset-group order
CORPUS_ORDER = ["CUAD", "Wikinews", "GDPR", "eCFR"]
TOOL_VERSIONS = {
    "python": "3.12", "spacy": "3.7.5", "spacy_model": "en_core_web_sm==3.7.1",
    "nltk": "3.9.1 (wordnet, omw-1.4)", "numpy": "1.26.4",
    "beautifulsoup4": "4.12.3 (lxml 5.3.0 parser)",
    "reserved_next_iteration": "sentence-transformers all-MiniLM-L6-v2, rank_bm25 "
                               "(NOT used for gold here; reserved for next-iteration "
                               "relation-alignment / retrieval)",
}
# data.py reads its raw/ snapshot relative to its own dir; the demo has no raw/ snapshot
# (it loads pre-built data), so ROOT points at the cwd and raw_manifest() reports 0 files.
ROOT = Path(".")

## Step 1 — Flatten & re-group the documents

`datasets[]` is **grouped by source corpus**, and each document is one example. We
flatten it to a list of document *rows* (capped at `MAX_DOCS_PER_CORPUS` per corpus),
then re-group by corpus in the fixed `CORPUS_ORDER` and sort within each group by
`doc_id` — exactly the grouping `data.py:main()` applies before writing the dataset.

In [ ]:
# flatten the corpus-grouped datasets[] into a flat list of document rows
rows = []
for group in data["datasets"]:
    rows.extend(group["examples"][:MAX_DOCS_PER_CORPUS])
logger.info(f"built {len(rows)} document rows")

# group by source corpus (stable order), sort within group by doc_id  [data.py:main()]
def corpus_of(source: str) -> str:
    return source.split(":")[0]

groups = defaultdict(list)
for r in rows:
    groups[corpus_of(r["metadata_source"])].append(r)
datasets = []
for c in CORPUS_ORDER:
    if groups.get(c):
        ex = sorted(groups[c], key=lambda r: r["metadata_doc_id"])
        datasets.append({"dataset": c, "examples": ex})
logger.info(f"datasets(by corpus)={[(d['dataset'], len(d['examples'])) for d in datasets]}")

## Step 2 — Recompute dataset statistics with `build_meta()` (verbatim from `data.py`)

`build_meta()` is copied **verbatim** from `data.py`. It is pure-Python aggregation
over the document rows (`Counter` / `statistics`) — **no model, no LLM** — producing
genre/source/quality counts, the crisp-subset size, the deterministic
`decidable_fraction` coverage proxy, and facts/char-length summaries. `raw_manifest()`
and `corpus_of()` are its helpers, also copied verbatim.

In [ ]:
def raw_manifest():
    man = {}
    cu = ROOT / "raw" / "cuad_data" / "CUADv1.json"
    if cu.exists():
        man["CUADv1.json"] = {"bytes": cu.stat().st_size,
                              "sha256_16": hashlib.sha256(cu.read_bytes()).hexdigest()[:16]}
    man["gdpr_html_files"] = len(glob.glob(str(ROOT / "raw" / "gdpr" / "art-*.html")))
    man["wikinews_article_files"] = len(glob.glob(str(ROOT / "raw" / "wikinews" / "article_*.json")))
    man["ecfr_xml_files"] = len(glob.glob(str(ROOT / "raw" / "ecfr" / "*.xml")))
    return man


def build_meta(rows):
    genres = Counter(r["metadata_genre"] for r in rows)
    quality = Counter(r["metadata_gold_quality"] for r in rows)
    qbyg = defaultdict(Counter); cbyg = Counter(); lic = Counter()
    relvocab = defaultdict(set); lens = []; nfacts = []; nents = []
    dec_all = []; dec_by_g = defaultdict(list); sub_all = []
    crisp_subset = 0
    for r in rows:
        gg = r["metadata_genre"]
        qbyg[gg][r["metadata_gold_quality"]] += 1
        cbyg[corpus_of(r["metadata_source"])] += 1
        lic[r["metadata_license"]] += 1
        for rel in r["metadata_relation_vocab"]:
            relvocab[gg].add(rel)
        lens.append(r["metadata_char_length"]); nfacts.append(r["metadata_num_facts"])
        nents.append(r["metadata_num_entities"])
        df = r["metadata_decidable_fraction"]
        dec_all.append(df); dec_by_g[gg].append(df)
        sub_all.append(r["metadata_decidable_subscores"])
        if r.get("metadata_crisp_subset"):
            crisp_subset += 1

    def _stat(xs):
        return {"min": round(min(xs), 4), "max": round(max(xs), 4),
                "mean": round(statistics.mean(xs), 4),
                "median": round(statistics.median(xs), 4)}
    decidable_summary = {
        "note": ("Deterministic coverage proxy (NO model). composite = mean(sentence_coverage, "
                 "entity_participation, char_coverage). Descriptive per-row feature (like "
                 "num_facts) so the experiment can rank/select the most-decidable documents; "
                 "NOT an experiment metric."),
        "overall": _stat(dec_all),
        "by_genre": {g: _stat(v) for g, v in dec_by_g.items()},
        "subscore_means_overall": {
            k: round(statistics.mean([s[k] for s in sub_all]), 4)
            for k in ("sentence_coverage", "entity_participation", "char_coverage")},
    }
    return {
        "name": "application_anchor",
        "n_documents": len(rows),
        "n_source_datasets": len(set(corpus_of(r["metadata_source"]) for r in rows)),
        "genre_counts": dict(genres),
        "source_dataset_counts": dict(cbyg),
        "gold_quality_counts": dict(quality),
        "gold_quality_by_genre": {k: dict(v) for k, v in qbyg.items()},
        "license_counts": dict(lic),
        "relation_vocab_by_genre": {k: sorted(v) for k, v in relvocab.items()},
        "crisp_subset_count": crisp_subset,
        "decidable_fraction": decidable_summary,
        "total_facts": sum(nfacts), "total_entities": sum(nents),
        "facts_per_doc": {"min": min(nfacts), "max": max(nfacts),
                          "mean": round(statistics.mean(nfacts), 2)},
        "char_length": {"min": min(lens), "max": max(lens),
                        "mean": round(statistics.mean(lens), 1),
                        "median": int(statistics.median(lens))},
        "determinism": {
            "seed": SEED, "tool_versions": TOOL_VERSIONS,
            "raw_cache_manifest": raw_manifest(),
        },
    }


meta = build_meta(rows)
logger.info(f"genres={meta['genre_counts']} quality={meta['gold_quality_counts']} "
            f"facts={meta['total_facts']} ents={meta['total_entities']}")
print(json.dumps({k: meta[k] for k in
      ["n_documents", "n_source_datasets", "genre_counts", "source_dataset_counts",
       "gold_quality_counts", "crisp_subset_count", "total_facts", "total_entities",
       "facts_per_doc", "char_length"]}, indent=2))

## Step 3 — Inspect a worked example & re-verify char-span provenance

Each example's `input` / `output` fields are JSON **strings**. We parse them and then
re-verify the dataset's core integrity claim exactly as `build/verify_dataset.py` does:

1. every entity `char_span` satisfies `document_text[s:e] == entity name`, and
2. every value-fact `tail` is a substring of its `provenance_char_span`

(the only legitimate non-substring tails are clause-presence **label** facts whose
provenance is the human-annotated clause span).

In [ ]:
for r in rows[:N_WORKED_EXAMPLES]:
    inp = json.loads(r["input"])      # input is a JSON STRING
    outp = json.loads(r["output"])    # output is a JSON STRING
    text = inp["document_text"]
    print("=" * 95)
    print(f"doc_id={inp['doc_id']}  genre={inp['genre']}  source={inp['source']}")
    print(f"gold_quality={r['metadata_gold_quality']}  crisp_subset={r['metadata_crisp_subset']}  "
          f"decidable_fraction={r['metadata_decidable_fraction']}")
    print("-" * 95)
    print("DOCUMENT (first 400 chars):")
    print("  " + text[:400].replace("\n", " ") + ("..." if len(text) > 400 else ""))
    print("-" * 95)
    print(f"ENTITIES ({len(inp['entities'])}) -- char_span verification (showing up to 8):")
    for e in inp["entities"][:8]:
        s, t = e["char_span"]
        ok = text[s:t] == e["name"]
        print(f"  [{'OK' if ok else 'XX'}] {e['type']:5} {e['name']!r}  span={e['char_span']}")
    all_ent_ok = all(text[e["char_span"][0]:e["char_span"][1]] == e["name"]
                     for e in inp["entities"])
    print(f"  -> all {len(inp['entities'])} entity spans verify: {all_ent_ok}")
    print("-" * 95)
    print(f"GOLD ATOMIC FACTS ({len(outp['gold_atomic_facts'])}):")
    for f in outp["gold_atomic_facts"]:
        s, t = f["provenance_char_span"]
        prov = text[s:t]
        print(f"  ({f['head']!r}, {f['relation']}, {f['tail']!r})")
        print(f"      provenance[{s}:{t}]={prov[:80]!r}  | tail in provenance: {f['tail'] in prov}")
    print()

## Step 4 — Summary table & visualizations

A compact per-genre table plus three charts over the loaded subset: documents per
genre (the leave-one-genre-out **fold**), the mean deterministic `decidable_fraction`
per genre (regulatory > news ≈ legal), and the distribution of gold atomic facts per
document.

In [ ]:
# --- per-genre summary table ---
genres = meta["genre_counts"]
dec_by_genre = {g: v["mean"] for g, v in meta["decidable_fraction"]["by_genre"].items()}
nfacts = [r["metadata_num_facts"] for r in rows]

print(f"{'genre':<12}{'docs':>6}{'crisp':>7}{'silver':>8}{'mean_decidable':>16}")
for g in genres:
    q = meta["gold_quality_by_genre"][g]
    print(f"{g:<12}{genres[g]:>6}{q.get('crisp', 0):>7}{q.get('silver', 0):>8}"
          f"{dec_by_genre.get(g, 0):>16.3f}")
print(f"\ncrisp_subset (CUAD legal): {meta['crisp_subset_count']}/{meta['n_documents']} docs")
print(f"total gold facts: {meta['total_facts']}  |  total entities: {meta['total_entities']}")

# --- charts ---
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(list(genres.keys()), list(genres.values()), color="steelblue")
ax[0].set_title("Documents per genre (fold)"); ax[0].set_ylabel("# docs")
ax[1].bar(list(dec_by_genre.keys()), list(dec_by_genre.values()), color="seagreen")
ax[1].set_title("Mean decidable_fraction by genre"); ax[1].set_ylim(0, 1)
ax[2].hist(nfacts, bins=range(min(nfacts), max(nfacts) + 2), color="indianred", align="left")
ax[2].set_title("Gold atomic facts per document"); ax[2].set_xlabel("# facts"); ax[2].set_ylabel("# docs")
plt.tight_layout(); plt.show()